# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 02: Training — Experiment 1 (SINetV2 on ACD1K)

**Author:** Yansong Jia (Experiment 1 Lead)  
**Dataset:** ACD1K only (military camouflage soldiers)  
**Model:** SINetV2 (lightweight encoder–decoder)

**Purpose:** Train a SINetV2 baseline on the ACD1K dataset at 352×352
resolution and export the best validation-checkpoint and training history
for unified final evaluation.

---

### Experiment 1 Design

| Setting | Value |
|---|---|
| Model | SINetV2 (custom PyTorch, binary output) |
| Training data | ACD1K train (748 images) |
| Validation data | ACD1K val (230 images, fixed splits) |
| Input resolution | **352 × 352** |
| Optimizer | AdamW |
| Loss | Binary cross-entropy + Dice |
| Early stopping | patience = 10 epochs (val mIoU) |
| LR sweep | 1e-4, 6e-5, 1e-5 |
| Outputs | `outputs/exp1/final/best_model.pth`, `history.json` |

This notebook deliberately **does not** run any final 200-image hold-out
evaluation or cross-experiment comparison. Those steps are handled by
Sepehr's final evaluation notebook.

---
## 1. Environment Setup

In [1]:
!rm -rf /content/AI-final-project
# Cell 1 — Clone repo
!git clone -b Yansong https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project

Cloning into 'AI-final-project'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 318 (delta 54), reused 80 (delta 42), pack-reused 220 (from 2)
Receiving objects: 100% (318/318), 52.84 MiB | 13.78 MiB/s, done.
Resolving deltas: 100% (141/141), done.
/content/AI-final-project


In [2]:
# Cell 2 — Install dependencies
!pip install -q -r requirements.txt kaggle

In [3]:
# (Optional) Cell 3 — Mount Google Drive to save outputs
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


---
## 2. Dataset Download (ACD1K only)

Experiment 1 uses only **ACD1K**. COD10K and CAMO are not required here.

In [4]:
# Cell 4 — Upload kaggle.json (run once per session)
import os
from google.colab import files

if not os.path.exists('/root/.config/kaggle/kaggle.json'):
    files.upload()  # select kaggle.json from your machine when prompted
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
    os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
    print('Kaggle credentials configured')
else:
    print('Kaggle credentials already configured')

Kaggle credentials already configured


In [5]:
# Cell 5 — Download ACD1K (~350 MB)
import os

acd_train = 'data/dataset-splitM/Training/images'
if not os.path.exists(acd_train):
    !kaggle datasets download \
        -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
        -p data/ --unzip
else:
    print('✅ ACD1K already exists, skipping download')

Dataset URL: https://www.kaggle.com/datasets/aalihhiader/military-camouflage-soldiers-dataset-mcs1k
License(s): CC0-1.0
100% 372M/372M [00:26<00:00, 14.5MB/s]



---
## 3. Dataset Verification (ACD1K paths only)

In [6]:
# Cell 6 — Verify ACD1K folder structure
import os

expected = [
    'data/dataset-splitM/Training/images',
    'data/dataset-splitM/Training/GT',
    'data/dataset-splitM/Testing/images',
    'data/dataset-splitM/Testing/GT',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'{status} — {p}')

✅ — data/dataset-splitM/Training/images
✅ — data/dataset-splitM/Training/GT
✅ — data/dataset-splitM/Testing/images
✅ — data/dataset-splitM/Testing/GT


In [7]:
# Cell 7 — Quick ACD1K-only sanity check
import torch
from pathlib import Path

import sys
sys.path.insert(0, str(Path.cwd() / 'src'))
from dataset import build_dataloader

train_loader = build_dataloader(
    'data/', condition='acd1k', split='train',
    batch_size=4, num_workers=2,
    oversample_acd1k=False, seed=42,
    splits_dir='splits/',
)
val_loader = build_dataloader(
    'data/', condition='acd1k', split='val',
    batch_size=4, num_workers=2,
    oversample_acd1k=False, seed=42,
    splits_dir='splits/',
)

batch = next(iter(train_loader))
print('Train batch image shape:', batch['image'].shape)
print('Train batch mask  shape:', batch['mask'].shape)
print('Train datasets:', set(batch['dataset']))

batch_val = next(iter(val_loader))
print('Val batch image shape  :', batch_val['image'].shape)
print('Val batch mask  shape  :', batch_val['mask'].shape)
print('Val datasets:', set(batch_val['dataset']))

  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=187
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=58
Train batch image shape: torch.Size([4, 3, 512, 512])
Train batch mask  shape: torch.Size([4, 1, 512, 512])
Train datasets: {'ACD1K'}
Val batch image shape  : torch.Size([4, 3, 512, 512])
Val batch mask  shape  : torch.Size([4, 1, 512, 512])
Val datasets: {'ACD1K'}


---
## 4. GPU Verification

In [8]:
# Cell 8 — Check GPU
!nvidia-smi

Thu Apr  9 06:52:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   33C    P0             99W /  700W |     527MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

---
## 5. Learning Rate Sweep (val mIoU on ACD1K)

We run three short training runs with different learning rates on ACD1K,
using validation mIoU to select the best LR for the final run.

- Run 1: `lr=1e-4`  
- Run 2: `lr=6e-5`  
- Run 3: `lr=1e-5`  

All runs write their history and (temporary) best_model checkpoints under
`outputs/exp1/sweep_lr*/`. Only the **final** run below writes to
`outputs/exp1/final/`.

In [16]:
# Cell 9 — Sweep run 1 (lr=1e-4)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 1e-4 --epochs 30 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1/sweep_lr1e4

Device: cuda
Config saved → outputs/exp1/sweep_lr1e4/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp1/sweep_lr1e4",
  "lr": 0.0001,
  "weight_decay": 0.0001,
  "epochs": 30,
  "batch_size": 8,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] ACD1K train: 748 samples, 93 batches
  [DataLoader] ACD1K val  : 230 samples, 29 batches

[Model] Building SINetV2...

[Training] 30 epochs, lr=0.0001, batch_size=8, patience=10
Ep   1/30 | train loss=1.3482 mIoU=0.4650 | val loss=1.2069 mIoU=0.4912 F1=0.3232 MAE=0.2461 ◀ best
Ep   2/30 | train loss=1.0629 mIoU=0.5174 | val loss=1.1019 mIoU=0.5196 F1=0.3974 MAE=0.2494 ◀ best
Ep   3/30 | train loss=0.9828 mIoU=0.5447 | val loss=1.0608 mIoU=0.5296 F1=0.3907 MAE=0.2303 ◀ best
Ep   4/30 | train loss=0.9445 mIoU=0.5565 |

In [17]:
# Cell 10 — Sweep run 2 (lr=6e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 6e-5 --epochs 30 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1/sweep_lr6e5

Device: cuda
Config saved → outputs/exp1/sweep_lr6e5/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp1/sweep_lr6e5",
  "lr": 6e-05,
  "weight_decay": 0.0001,
  "epochs": 30,
  "batch_size": 8,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] ACD1K train: 748 samples, 93 batches
  [DataLoader] ACD1K val  : 230 samples, 29 batches

[Model] Building SINetV2...

[Training] 30 epochs, lr=6e-05, batch_size=8, patience=10
Ep   1/30 | train loss=1.4567 mIoU=0.4442 | val loss=1.2794 mIoU=0.4783 F1=0.2937 MAE=0.2459 ◀ best
Ep   2/30 | train loss=1.1381 mIoU=0.5013 | val loss=1.1521 mIoU=0.5071 F1=0.3572 MAE=0.2440 ◀ best
Ep   3/30 | train loss=1.0316 mIoU=0.5299 | val loss=1.0949 mIoU=0.5232 F1=0.3945 MAE=0.2412 ◀ best
Ep   4/30 | train loss=0.9838 mIoU=0.5458 | v

In [18]:
# Cell 11 — Sweep run 3 (lr=1e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 1e-5 --epochs 30 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1/sweep_lr1e5

Device: cuda
Config saved → outputs/exp1/sweep_lr1e5/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp1/sweep_lr1e5",
  "lr": 1e-05,
  "weight_decay": 0.0001,
  "epochs": 30,
  "batch_size": 8,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] ACD1K train: 748 samples, 93 batches
  [DataLoader] ACD1K val  : 230 samples, 29 batches

[Model] Building SINetV2...

[Training] 30 epochs, lr=1e-05, batch_size=8, patience=10
Ep   1/30 | train loss=1.8819 mIoU=0.3925 | val loss=1.7001 mIoU=0.4169 F1=0.2383 MAE=0.3084 ◀ best
Ep   2/30 | train loss=1.5230 mIoU=0.4270 | val loss=1.4874 mIoU=0.4411 F1=0.2733 MAE=0.2935 ◀ best
Ep   3/30 | train loss=1.3794 mIoU=0.4510 | val loss=1.3754 mIoU=0.4629 F1=0.3039 MAE=0.2772 ◀ best
Ep   4/30 | train loss=1.2909 mIoU=0.4698 | v

---
## 6. Inspect Sweep Results and Choose Best LR

We inspect the saved `history.json` files and select the LR with the
highest validation mIoU on ACD1K.

In [19]:
# Cell 12 — Compare sweep histories
import json

def best_miou(path):
    with open(path) as f:
        hist = json.load(f)
    row = max(hist, key=lambda x: x['val_mIoU'])
    return row['epoch'], row['val_mIoU'], row['val_F1'], row['val_MAE']

runs = {
    'lr=1e-4': 'outputs/exp1/sweep_lr1e4/final/history.json',
    'lr=6e-5': 'outputs/exp1/sweep_lr6e5/final/history.json',
    'lr=1e-5': 'outputs/exp1/sweep_lr1e5/final/history.json',
}

for label, path in runs.items():
    try:
        ep, miou, f1, mae = best_miou(path)
        print(f"{label}: best epoch={ep}, val mIoU={miou:.4f}, F1={f1:.4f}, MAE={mae:.4f}")
    except FileNotFoundError:
        print(f"{label}: history.json not found at {path}")

lr=1e-4: best epoch=30, val mIoU=0.5981, F1=0.4866, MAE=0.1716
lr=6e-5: best epoch=26, val mIoU=0.5865, F1=0.4739, MAE=0.1825
lr=1e-5: best epoch=28, val mIoU=0.5322, F1=0.3849, MAE=0.2212


> After inspecting the printed results above, choose the best learning
> rate (e.g., 6e-5) and use it for the final training run below.

---
## 7. Final Training Run (best LR)

We now run a longer training with early stopping using the best learning
rate from the sweep. This run writes its artifacts to
`outputs/exp1/final/` and is the only checkpoint consumed by the
project's unified final evaluation.

In [22]:
# Cell 13 — Final Experiment 1 training (update --lr if sweep differs)
!PYTHONPATH=/content/AI-final-project/src \
 python src/engine_exp1.py \
    --lr 1e-4 --epochs 60 --batch_size 8 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp1

Device: cuda
Config saved → outputs/exp1/config.json
{
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp1",
  "lr": 0.0001,
  "weight_decay": 0.0001,
  "epochs": 60,
  "batch_size": 8,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] ACD1K train: 748 samples, 93 batches
  [DataLoader] ACD1K val  : 230 samples, 29 batches

[Model] Building SINetV2...

[Training] 60 epochs, lr=0.0001, batch_size=8, patience=10
Ep   1/60 | train loss=1.3676 mIoU=0.4594 | val loss=1.2352 mIoU=0.4767 F1=0.2621 MAE=0.2360 ◀ best
Ep   2/60 | train loss=1.0613 mIoU=0.5180 | val loss=1.0860 mIoU=0.5253 F1=0.3817 MAE=0.2291 ◀ best
Ep   3/60 | train loss=0.9781 mIoU=0.5464 | val loss=1.0962 mIoU=0.5227 F1=0.4056 MAE=0.2474
Ep   4/60 | train loss=0.9435 mIoU=0.5568 | val loss=1.0866 mIoU=0.5103 F1

---
## 8. Verify Outputs and Handoff to Final Evaluation

Experiment 1's responsibility ends with producing the best validation
checkpoint and training history. The unified 200-image hold-out
evaluation is handled elsewhere.

In [23]:
# Cell 14 — Confirm best_model.pth and history.json
import os

paths = [
    'outputs/exp1/final/best_model.pth',
    'outputs/exp1/final/history.json',
]
for p in paths:
    print('✅' if os.path.exists(p) else '❌ NOT FOUND', p)

✅ outputs/exp1/final/best_model.pth
✅ outputs/exp1/final/history.json


If both files above exist and look reasonable, Experiment 1 is complete.

- `outputs/exp1/final/best_model.pth` — SINetV2 weights + metadata  
- `outputs/exp1/final/history.json` — per-epoch train/val metrics

These artifacts can now be passed to the project's final evaluation
notebook for unified testing on the 200-image hold-out set.

---
## 9. Sync Notebook + Exp1 Outputs to GitHub

Run this section after training completes if you want to push the latest
`notebooks/02_train_exp1_yansong.ipynb` and all `outputs/exp1/` artifacts
back to the GitHub repository.

In [29]:
# Cell 15 — Commit and push Exp1 notebook + outputs to GitHub
# Note: In Colab, you need authentication (PAT) to push.

import os
from getpass import getpass
# Optional: configure identity for this Colab runtime only
!git config user.email "ys.jia@outlook.com"
!git config user.name "y4medi"

# Ensure we are on the Yansong branch
!git checkout Yansong

# Add updated Exp1 notebook and all Exp1 outputs
!git add notebooks/02_train_exp1_yansong.ipynb
!git add outputs/exp1/

# Commit if there are staged changes
status = !git status --porcelain
if len(status) == 0:
    print("No changes to commit.")
else:
    !git commit -m "Exp1: update SINetV2 notebook and outputs"

#  SAFE PUSH (NO TOKENS IN CODE)
print("\nEnter your GitHub PAT when prompted (input is hidden).")
token = getpass("GitHub PAT: ")
!git pull origin Yansong --rebase --autostash
# Push using temporary authenticated URL (no credentials saved)
repo_url = f"https://{token}@github.com/bing-er/AI-final-project.git"
!git push {repo_url} Yansong

Already on 'Yansong'
Your branch is ahead of 'origin/Yansong' by 1 commit.
  (use "git push" to publish your local commits)
No changes to commit.

Enter your GitHub PAT when prompted (input is hidden).
GitHub PAT: ··········
From https://github.com/bing-er/AI-final-project
 * branch            Yansong    -> FETCH_HEAD
   dfdf982..5d20278  Yansong    -> origin/Yansong
Already up to date.
Everything up-to-date
